In [1]:
import numpy as np, pickle, os, time as time_lib, sys, csv
import matplotlib.pyplot as plt

In [2]:
sys.path.append('/home/filip/Git_Code/')

from demler_tools.file_manager import path_management, file_management, io
from demler_tools.file_manager import path_management, file_management_local_backend
from demler_tools.file_manager import request_creation                                                                               
from demler_tools.file_manager import file_management,cluster_backend_tools
from demler_tools.parallelization import local_run_parallel, cluster_interface

path_management.initialize(project_name = 'psBQP-keldysh')

import data_support as ds


Successfully initialized path management with following parameters:
     username:                fmarijanovic
     project name:            psBQP-keldysh
     controlling machine:     laptop
     library version:         v2.3.0-beta-17-g6b0081d
     auto SSH transfer:       True
     default save location:   /home/filip/Documents/Research/data_files/
     LTS mount point:         /home/filip
     email domain:            phys.ethz.ch
     cluster OS:              Ubuntu
     cluster partition:       work
     SSH host file:           /home/filip/.ssh/known_hosts
     SSH key file:            /home/filip/.ssh/id_ed25519_euler
     default modules:         {'CentOS': 'gcc/8.2.0 python/3.11.2', 'Ubuntu': 'stack/2024-06 python/3.11.6'}


In [3]:
crt_controlling_machine = 'laptop'
crt_simulations_machine = 'cluster_euler'

In [4]:
print(np.__version__)
print(path_management.CRT_PATH_CONVENTION)

1.24.2
v3


## Cluster run

In [5]:
crt_simulations_machine = 'cluster_euler'


### Making a job request -- looping vector potential values

In [35]:
simulation_type = 'Normal state non-linearity'
simulation_desc = 'Strong pulse response -- lower damping -- above Tc '


_loop_kwargs = 'field_params'
evolution_timesteps = 1001
initial_state_timestamp = '1782219875' #1782219990 -- eta = 0.2; 1782219875 -- eta = 0.1
initial_state_index = 22
grid_parameters = {'time_sampling': 2001, 'time_duration': 2 * np.pi * 10}
#vector_potential = np.ones(evolution_timesteps) * 0.0
#temp_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8]
system_parameters = {'critical_temperature': 1, 'temperature': 0.5, 'eta': 0.1}
#for temp in temp_list:
#    system_parameters += []

occupation_tracking_every_n = None

save_full_state = False

field_params = []


#* FWHMS
#FWHMs = np.linspace(10.0,10.0,1,endpoint=True)
#FWHMs = np.linspace(1.0,1.0,1,endpoint=True)
#FWHMs = np.linspace(40.0,35.0,2,endpoint=True)
FWHMs = [10.0,5.0,1.0]
#* Amplitudes
amplitudes = np.linspace(0.1,1.0,10,endpoint=True)
#amplitudes = np.linspace(0.1,0.5,5,endpoint=True)
#amplitudes = [0.05]
#* Frequencies
#frequencies = (np.linspace(0.5,1.5,5,endpoint=True)) * 1.6/2/np.pi
frequencies = [0.0]

field_type = 'gaussian'

for fwhm in FWHMs:
    for amplitude in amplitudes:
        for frequency in frequencies:
            field_params += [{'amplitude': amplitude, 'FWHM': fwhm, 'frequency': frequency, 'phase': 0.0}] #, 'frequency': 2.0 * 1/2/np.pi,'phase': np.pi/2}]


#for fwhm in FWHMs:
#    field_params += [{'amplitude': 0.1, 'FWHM': fwhm}]


circuit_params = None


### Making a job request -- looping temperatures

In [11]:
simulation_type = 'Undriven thermal equilibrium'
simulation_desc = 'Equilibrium temperature result -- strong damping'

_loop_kwargs = 'system_parameters'
evolution_timesteps = 300
initial_state_timestamp = None #1782219990 #'1782219990' #'1781702763' #'1780759388' #'1780908007' #'1780759388'
initial_state_index = 8
grid_parameters = {'time_sampling': 501, 'time_duration': 2 * np.pi * 5}
#temp_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8]
#temp_list = [0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.9,0.95,1.0,1.05,1.1,1.3,1.5,2.0]
temp_list = np.linspace(0.1,1.2,12,endpoint=True)
#temp_list = [0.1]
#for temp in temp_list:
#    system_parameters += []
system_parameters = []
#vector_potential = []
occupation_tracking_every_n = None

save_full_state = True 

for temp in temp_list:
    system_parameters += [{'critical_temperature': 1, 'temperature': temp, 'eta': 0.2}]

field_type = None
field_params = None

circuit_params = None


### Making a job request -- looping incoming pulses

In [ ]:
simulation_type = 'Current pulse response'
simulation_desc = 'Different pulse durations for current pulse response -- weak damping -- zero contact'


_loop_kwargs = 'field_params'
evolution_timesteps = 1001
initial_state_timestamp = '1782219875' #1782219990 -- eta = 0.2; 1782219875 -- eta = 0.1
initial_state_index = 2
grid_parameters = {'time_sampling': 2001, 'time_duration': 2 * np.pi * 10}
#vector_potential = np.ones(evolution_timesteps) * 0.0
#temp_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8]
system_parameters = {'critical_temperature': 1, 'temperature': 0.2, 'eta': 0.1}
#for temp in temp_list: 
#    system_parameters += []

field_type = 'current_gaussian'
occupation_tracking_every_n = None

save_full_state = False

field_params = []

vec_potential_mags = [0.1,0.5]

tc_rescaling = 1
#* If rescaling T_c need to rescale T/T_c as well! -- for now ignoring that!    
T_c = 15.5 / tc_rescaling
t_pulse = 0.020 * T_c * 2 * np.pi

critical_current_ts = '1782982330' #'1782292287' eta = 0.1 T = 0.5, '1782982330' eta = 0.1, T = 0.2, eta = 0.2  '1782220134'
critical_current = ds.extract_critical_current(job_run_timestamp = critical_current_ts, critical_current_timestamp = critical_current_ts, n_average = 50)

circuit_params = {'Z_T': 50.0, 'R_n': 170.0, 'J_DP_prime': critical_current,'R_c': 0.0}

#* FWHMS
#FWHMs = np.linspace(10.0,10.0,1,endpoint=True)
#FWHMs = np.linspace(5.0,1.0,5,endpoint=True)
#FWHMs = np.linspace(30.0,10.0,5,endpoint=True)
FWHMs = [t_pulse/2] #, t_pulse, t_pulse / 2]
#* Amplitudes
amplitudes = np.linspace(0.05,20.0,50,endpoint=True)
#amplitudes = np.linspace(0.1,0.5,5,endpoint=True)
#amplitudes = [0.05]
#* Frequencies
#frequencies = (np.linspace(0.2,2.0,50,endpoint=True)) * 1.6/2/np.pi
frequencies = [0.0]
#field_type = 'gaussian'

for fwhm in FWHMs:
    for amplitude in amplitudes:
        for frequency in frequencies:
            field_params += [{'amplitude': amplitude, 'FWHM': fwhm, 'frequency': frequency, 'phase': 0.0}] #, 'frequency': 2.0 * 1/2/np.pi,'phase': np.pi/2}]

#for fwhm in FWHMs:
#    field_params += [{'amplitude': 0.1, 'FWHM': fwhm}]

#current_function_gaussian = lambda t: np.exp(-(t - 0)**2 * np.log(2)/(t_pulse)**2) 



EXTRACTING CRITICAL CURRENT

Job run timestamp:           1782982330
Critical current timestamp:  1782982330

----------------------------------------------------------------------
LOADING DATA
----------------------------------------------------------------------

Number of jobs in critical current sweep: 21

----------------------------------------------------------------------
PARAMETER COMPARISON
----------------------------------------------------------------------

✓ All parameters match (excluding field_params and num_timesteps)

----------------------------------------------------------------------
RUN PARAMETERS (from critical_current_timestamp)
----------------------------------------------------------------------

num_timesteps: 800
system_parameters:
  critical_temperature: 1
  temperature: 0.2
  eta: 0.1
initial_state_timestamp: 1782219875
initial_state_index: 4
grid_parameters:
  time_sampling: 2001
  time_duration: 62.83185307179586
field_type: constant
field_params:
  

### Creating jobs

In [36]:
crt_run_parameters = file_management.single_variable_kwargs_array(_loop_tag=_loop_kwargs, num_timesteps = evolution_timesteps, system_parameters = system_parameters, initial_state_timestamp = initial_state_timestamp, initial_state_index=initial_state_index, grid_parameters=grid_parameters, field_type=field_type, field_params=field_params, track_every_n=occupation_tracking_every_n, circuit_params = circuit_params,save_full_state = save_full_state)

request_creation.create_request_folder_any_machine(                                                                                  
      calculation_type = simulation_type,                                                                                              
      machine_identifier = crt_simulations_machine,                                                                                    
      kwargs_array = crt_run_parameters,                                                                                               
      job_memory_unit = 'G',                                                                                                           
      job_memory_value = 12,                                                                                                          
      job_time = (0,36,0,0),                                                                                                           
      task_summary = simulation_desc,                                                                                                  
      verbose = True,
      cpus_per_task = 1,
      # NEW v2-style parameters:
      solver_method_file = 'code_run.py',   
      solver_method_name = 'evolve_keldysh_state',
      #cluster_backend = 'v2.2.0-beta'  # Or whichever version exists on cluster
  )

Created request folder: 1783015498


In [9]:
file_management.display_unattempted_folders(machine_identifier = crt_simulations_machine, folder_summaries = True)

Directory 1782657005 has not been attempted on cluster.

      Human-readable file created at Sun Jun 28 16:30:05 2026, using demler_tools v2.3.0-beta-17-g6b0081d
      Path convention: v3
      Created by user: fmarijanovic
      Project name: psBQP-keldysh
      
      Calculation type: Current pulse response
      Number of jobs to run: 1
      
      Task summary: Current pulse quench
      
      Current folder state: unattempted
      
      Calculation log:





In [37]:
cluster_interface.push_unattempted_folders(machine_identifier = crt_simulations_machine)

Folder 1783015498 submitted successfully.


In [ ]:
file_management.display_all_folders(machine_identifier =        crt_simulations_machine, folder_summaries = True)

Directory with timestamp 1780988799 has status: submitted.

      Human-readable file created at Tue Jun  9 09:06:39 2026, using demler_tools v2.3.0-beta-17-g6b0081d
      Path convention: v3
      Created by user: fmarijanovic
      Project name: psBQP-keldysh
      
      Calculation type: Equilibration
      Number of jobs to run: 24
      
      Task summary: Computing the equilibrium state for future reference
      
      Current folder state: submitted
      
      Calculation log:
      [Tue Jun  9 09:06:55 2026] Updated directory state to the following: submitted
      [Tue Jun  9 09:06:56 2026] Submitted folder job to cluster with controller cluster_files/cluster_control_0_main.sh and under ID: 2660482
      [Tue Jun  9 09:06:56 2026] Submitted automatic postprocess job to cluster with controller cluster_files/cluster_control_0_post.sh and under ID: 2660484



Directory with timestamp 1780988780 has status: submitted.

      Human-readable file created at Tue Jun  9 09:06:20 

In [1]:
a = [1,2,3]
print(a[::2])

[1, 3]
